# 17. Select the augmented style and text blend

Owner: Jalil-g (reassigned from Gtpatel02 on November 10, 2026). Issue #17. Depends on completed #11, #14, #15.

## Goal

#15 picked a style + text + NB blend, but its style arm breaks on collapsed whitespace. Here we swap in the #14 `plus_collapse` style arm, rerun the same six-candidate grid on internal rows, and confirm the saved pick on frozen validation.

## Rule (fixed before running)

**Components.** All fit on internal 6400 for selection, then refit on the 8000 development rows for confirmation.
- Style: #13 HistGB on the 147 rich-style features, trained with #14 `plus_collapse` copies (original weight 1, collapsed copy weight .25). Score = logit of P(AI) clipped to [1e-7, 1-1e-7], standardized on eligible unlabeled.
- Old soft text (#11/#15): word1-3 + case char2-5 LR C16 on all fit rows, and case char2-5 SVM C1 on fit rows without a literal blank line. Each margin standardized on eligible unlabeled (specialist reference also no-blank). Mean of the two + 2 * literal blank flag (active only when no human fit row has `\n\n`), then outer standardization on eligible unlabeled.
- NB transfer (#15 treatment, unchanged): normalized binary NB word1-3 + char3-5 LR C1. Pseudo labels come from the original #13 HistGB teacher at confidence .995, pseudo rows weight .25. Margin standardized on eligible unlabeled. The teacher stays the original style model, so the only change from #15 is the style arm.

**Grid.** Same six weight vectors as #15 over (style, old soft, NB): hist, hist75_old25, hist50_old50, hist75_nb25, hist50_old25_nb25, old_soft. Final score = sigmoid of the weighted standardized scores. This is score averaging, not probability averaging. No test-batch normalization.

**Selection.** On internal 1600: eligible if remove-blank AUROC and collapse AUROC are each >= the pure augmented hist value - .002. Pick max original AUROC. Grid order breaks exact ties, so ties go to pure hist. wrap80 is measured but not used. The 50/50 candidate is in the grid but not forced. The choice is saved before validation is touched.

**Also measured, not used for selection.** The same grid with the original #13 style arm, to check we reproduce #15. On validation, every candidate is scored for both grids as a descriptive table only.

**Caveats known up front.** The style arm trains on collapsed copies, and collapse is a guardrail view, so part of its collapse gain is by design. Validation was reused before (#4, #8, #9, #13, #14, #15), so it is optimistic. Nothing here is a Kaggle score.

## Setup

Helpers come from notebooks 10, 13, and 08 in helpers-only mode, so nothing there fits or writes. #15 has no helpers-only mode, so its arm code is copied below and checked by reproducing its numbers. Everything runs with one thread.

In [ ]:
import os, json, time, gc, textwrap, unicodedata, warnings
from contextlib import contextmanager
from pathlib import Path
import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import vstack
from scipy.special import expit, logit
from IPython import get_ipython
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from threadpoolctl import threadpool_limits
os.environ["EXPANDED_TEXT_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./10_expanded_text.ipynb")
os.environ["RICH_STYLE_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./13_rich_style.ipynb")
os.environ["ERROR_ANALYSIS_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./08_error_analysis.ipynb")
# Keep the #08 diagnostic settings as they are; only its handoff pointer is not ours.
DIAG_CONFIG = {k: CONFIG[k] for k in ["measures", "bins", "threshold", "bootstrap", "cases_per_type", "case_order"]}

NAME = "17_final_blend"
COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=3600 notebooks/17_final_blend.ipynb"
STYLE_HIST_SETTINGS = {"max_iter": 250, "learning_rate": .06, "max_leaf_nodes": 15, "min_samples_leaf": 30,
                       "l2_regularization": 5., "early_stopping": False, "random_state": 42}
COPY_WEIGHT = .25
STYLE_COPIES = ["collapse"]  # #14 plus_collapse
BLEND_GRID = {"hist": [1., 0., 0.], "hist75_old25": [.75, .25, 0.], "hist50_old50": [.5, .5, 0.],
              "hist75_nb25": [.75, 0., .25], "hist50_old25_nb25": [.5, .25, .25], "old_soft": [0., 1., 0.]}
STRESS = ["original", "remove_blank", "collapse", "wrap80"]
GUARD_VIEWS, GUARD_SLACK = ["remove_blank", "collapse"], .002
CONFIG = {
    "components": ["style_aug", "old_soft", "nb_transfer"],
    "style": {"histgb": STYLE_HIST_SETTINGS, "copies": STYLE_COPIES, "copy_weight": COPY_WEIGHT,
              "score": "logit(clip(P(AI), 1e-7, 1-1e-7)), standardized on eligible unlabeled"},
    "old_soft": {"word": "word1-3 + case char2-5 LR C16, all fit rows", "specialist": "case char2-5 SVM C1, fit rows without literal blank line",
                 "flag": "literal two consecutive newlines; active only when zero human fit rows have it", "offset": 2.,
                 "score": "mean of standardized margins + active flag*2; outer standardization on eligible unlabeled",
                 "specialist_reference": "eligible unlabeled rows without literal blank lines"},
    "nb_transfer": {"features": "normalized text; binary CountVectorizer word1-3 max 120000 + char3-5 max 180000, min_df 2, float32",
                    "normalization": "NFKC; canonical quotes/dashes/ellipsis; remove Cf; collapse whitespace; lowercase",
                    "ratio": "NB log ratio alpha 1 on verified + assigned pseudo labels", "classifier": "LR C1 liblinear max_iter 3000 tol 1e-4 seed 42",
                    "teacher": "original #13 HistGB (no copies) on the same fit rows", "confidence": .995, "pseudo_weight": .25,
                    "score": "margin standardized on eligible unlabeled"},
    "blend_grid": BLEND_GRID, "stress": STRESS,
    "stress_definitions": {"remove_blank": "drop whitespace-only lines, join with \\n (#15)",
                           "collapse": "' '.join(text.split()) (#14/#15)",
                           "wrap80": "split on blank lines, textwrap.fill width 80, break_long_words=False, break_on_hyphens=False, join with blank line (#14)"},
    "selection": "on internal 1600: eligible if remove_blank and collapse each >= pure augmented hist - .002; max original AUROC; grid order breaks exact ties (pure hist first); wrap80 measured only; saved before validation",
    "guard_views": GUARD_VIEWS, "guard_slack": GUARD_SLACK,
    "reproduction": "same grid with original #13 style arm, compared to #15 internal and validation numbers; not used for selection",
    "score_mapping": "sigmoid of weighted standardized scores; calibration measured, not assumed",
    "internal": {"fit_rows": 6400, "score_rows": 1600, "seed": 42, "how": "train_test_split(test_size=.2, stratify=label, random_state=42) on manifest training order, then sorted indices (#15)"},
    "diagnostics": DIAG_CONFIG, "threads": "threadpool_limits(limits=1) for every fit and score",
}
START = time.perf_counter()

## Checks

Notebook 02 checks data bytes and the frozen split. We check the internal split matches both #14 and #15, that the dependency records still say what we copied, and that the stress views keep the word sequence.

In [ ]:
parts = load_partitions()
train, validation, reference = parts["train"], parts["validation"], parts["unlabeled"]
test = read_raw("test.csv")
validate_ids(test)
require(len(test) == 3000 and not set(test.id) & (set(train.id) | set(validation.id)), "Unexpected test IDs")
a, b = train_test_split(np.arange(len(train)), test_size=.2, stratify=train.label, random_state=42)
a, b = np.sort(a), np.sort(b)
internal_train, internal_score = train.iloc[a].reset_index(drop=True), train.iloc[b].reset_index(drop=True)
require(len(internal_train) == 6400 and internal_score.label.value_counts().to_dict() == {0: 800, 1: 800}, "Bad internal split")

def latest_completed(folder):
    rows = [(p, json.loads(p.read_text())) for p in sorted((ROOT / "results" / folder).glob("2*Z.json"))]
    rows = [(p, r) for p, r in rows if r.get("status") == "completed"]
    require(bool(rows), f"Completed {folder} record is required")
    return rows[-1]

dep11_path, dep11 = latest_completed("11_formatting_rule")
dep13_path, dep13 = latest_completed("13_rich_style")
dep14_path, dep14 = latest_completed("14_style_augmentation")
dep15_path, dep15 = latest_completed("15_normalized_transfer")
require(dep13["config"]["schema_sha256"] == SCHEMA_SHA256 and dep13["config"]["candidates"]["histgb"] == STYLE_HIST_SETTINGS, "#13 style changed")
require(dep14["metrics"]["selected"] == "plus_collapse" and dep14["config"]["presets"]["plus_collapse"] == STYLE_COPIES, "#14 pick changed")
require(dep14["config"]["copy_weight"] == COPY_WEIGHT and dep14["config"]["histgb"] == STYLE_HIST_SETTINGS, "#14 settings changed")
require(dep15["config"]["blend_grid"] == BLEND_GRID and dep15["config"]["teacher"] == STYLE_HIST_SETTINGS, "#15 grid changed")
internal_hash = {"fit": value_hash(internal_train.id.tolist()), "score": value_hash(internal_score.id.tolist())}
require(internal_hash == dep15["config"]["internal_id_sha256"], "Internal split differs from #15")
require(value_hash(sorted(internal_train.id)) == dep14["config"]["inner_split"]["fit_ids_sha256"], "Internal split differs from #14")
CONFIG["internal_id_sha256"] = internal_hash
CONFIG["reference"] = {"rows": len(reference), "ordered_id_sha256": value_hash(reference.id.tolist())}
CONFIG["dependencies"] = {k: {"record": str(p.relative_to(ROOT)), "record_sha256": file_hash(p)}
                          for k, p in [("11", dep11_path), ("13", dep13_path), ("14", dep14_path), ("15", dep15_path)]}
CONFIG["helper_source_sha256"] = {n: value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks" / n).read_text())["cells"]])
                                  for n in ["08_error_analysis.ipynb", "10_expanded_text.ipynb", "13_rich_style.ipynb", "15_normalized_transfer.ipynb"]}
CONFIG["notebook_source_sha256"] = value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/17_final_blend.ipynb").read_text())["cells"]])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / RUN_ID
private.mkdir(parents=True, exist_ok=False)
print(f"Train {len(train)}, validation {len(validation)}, eligible unlabeled {len(reference)}, test {len(test)}")
print("Internal split matches #14 and #15. Dependency settings match.")

## Method

Arm code is copied from #15 (`NormalizedLexicalModel` nb path, `SoftTextArm`, `TransferBlend`). The style arm adds the #14 copies. Features and NB ratios fit only on the current fit rows; reference statistics use eligible unlabeled text only.

In [ ]:
QUOTE_MAP = {ord(c): "'" for c in "\u2018\u2019\u201a\u201b"}
QUOTE_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f"})
QUOTE_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})
QUOTE_MAP[ord("\u2026")] = "..."
def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(QUOTE_MAP)
    text = "".join(c for c in text if unicodedata.category(c) != "Cf")
    return " ".join(text.split()).lower()

def literal_blank_flags(texts):
    return np.asarray(["\n\n" in str(t) for t in texts], dtype=bool)

def wrap_view(text, width):
    chunks = [c for c in PARAGRAPH_SPLIT.split(str(text)) if c.strip()]
    return "\n\n".join(textwrap.fill(c, width=width, break_long_words=False, break_on_hyphens=False) for c in chunks)

def stress_texts(texts, variant):
    if variant == "original": return [str(t) for t in texts]
    if variant == "remove_blank": return ["\n".join(line for line in str(t).split("\n") if line.strip()) for t in texts]
    if variant == "collapse": return [" ".join(str(t).split()) for t in texts]
    if variant == "wrap80": return [wrap_view(t, 80) for t in texts]
    raise ValueError("Unknown stress view")

class StyleArm:
    """#13 HistGB on rich-style features. copies=[] is the #13/#15 arm; ["collapse"] is #14 plus_collapse."""
    def __init__(self, copies):
        self.copies = list(copies)
    def fit(self, texts, labels, reference_texts):
        texts, labels = list(texts), np.asarray(labels)
        views = [texts] + [stress_texts(texts, v) for v in self.copies]
        X, y = np.vstack([rich_style_features(v) for v in views]), np.concatenate([labels] * len(views))
        # Copies at .25 each. No sample_weight without copies, exactly like #13 and #15.
        weights = np.concatenate([np.ones(len(labels))] + [np.full(len(labels), COPY_WEIGHT)] * len(self.copies)) if self.copies else None
        self.model_ = HistGradientBoostingClassifier(**STYLE_HIST_SETTINGS).fit(X, y, sample_weight=weights)
        self.classes_ = self.model_.classes_
        raw = self.raw_margin(reference_texts)
        self.statistics_ = {"mean": float(raw.mean()), "std": float(raw.std())}
        require(self.statistics_["std"] > 1e-12, "Degenerate style reference")
        return self
    def probabilities(self, texts): return class1_probabilities(self.model_, rich_style_features(list(texts)))
    def raw_margin(self, texts): return logit(np.clip(self.probabilities(texts), 1e-7, 1 - 1e-7))
    def decision_function(self, texts): return (self.raw_margin(texts) - self.statistics_["mean"]) / self.statistics_["std"]

def nb_features():
    shared = {"min_df": 2, "dtype": np.float32, "lowercase": False, "binary": True}
    return FeatureUnion([("word", CountVectorizer(ngram_range=(1, 3), max_features=120000, **shared)),
                         ("char", CountVectorizer(analyzer="char", ngram_range=(3, 5), max_features=180000, **shared))], n_jobs=1)

class NBTransferModel:
    """#15 nb treatment: vocabulary from verified rows only; NB ratio and LR also see quarter-weight pseudo rows."""
    def fit(self, texts, labels, pseudo_texts, pseudo_labels):
        self.features_ = nb_features()
        X = self.features_.fit_transform([normalize_text(t) for t in texts])
        self.vocabulary_sha256_ = value_hash(self.features_.get_feature_names_out().tolist())
        Xp = self.features_.transform([normalize_text(t) for t in pseudo_texts])
        X = vstack([X, Xp], format="csr")
        y = np.concatenate([np.asarray(labels), np.asarray(pseudo_labels)])
        weights = np.concatenate([np.ones(len(texts)), np.full(len(pseudo_texts), .25)])
        self.ratio_ = NBLogRatio(alpha=1.).fit(X, y)
        self.classifier_ = LogisticRegression(C=1., solver="liblinear", max_iter=3000, random_state=42, tol=1e-4)
        self.classifier_.fit(self.ratio_.transform(X), y, sample_weight=weights)
        self.classes_ = self.classifier_.classes_
        return self
    def decision_function(self, texts):
        return self.classifier_.decision_function(self.ratio_.transform(self.features_.transform([normalize_text(t) for t in texts])))

class SoftTextArm:
    def fit(self, texts, labels, reference_texts):
        texts = pd.Series(list(texts)); labels = np.asarray(labels); keep = ~literal_blank_flags(texts)
        self.rule_active_ = not bool(literal_blank_flags(texts[labels == 0]).any())
        require(set(labels[keep]) == {0, 1}, "No-blank specialist needs both classes")
        self.word_ = make_expanded_model({"view": "word_char", "learner": "lr", "C": 16.}).fit(texts, labels)
        self.specialist_ = make_expanded_model({"view": "casechar", "learner": "svm", "C": 1.}).fit(texts[keep], labels[keep])
        reference_texts = pd.Series(list(reference_texts))
        self.word_stats_ = margin_reference(self.word_, reference_texts)
        self.specialist_stats_ = margin_reference(self.specialist_, reference_texts[~literal_blank_flags(reference_texts)])
        raw = self.raw_margin(reference_texts); self.outer_ = {"mean": float(raw.mean()), "std": float(raw.std())}
        require(self.outer_["std"] > 1e-12, "Degenerate old text reference")
        self.fit_rows_, self.specialist_rows_ = len(texts), int(keep.sum())
        return self
    def raw_margin(self, texts):
        score = .5 * (standardized_margin(self.word_, texts, self.word_stats_) + standardized_margin(self.specialist_, texts, self.specialist_stats_))
        return score + 2. * literal_blank_flags(texts) * self.rule_active_
    def decision_function(self, texts): return (self.raw_margin(texts) - self.outer_["mean"]) / self.outer_["std"]

class TransferBlend:
    classes_ = np.array([0, 1])
    def __init__(self, style, old, nb, nb_stats, weights):
        self.style, self.old, self.nb, self.nb_stats, self.weights = style, old, nb, nb_stats, np.asarray(weights)
    def components(self, texts):
        return np.column_stack([self.style.decision_function(texts), self.old.decision_function(texts), standardized_margin(self.nb, texts, self.nb_stats)])
    def decision_function(self, texts): return self.components(texts) @ self.weights
    def predict_proba(self, texts):
        p = expit(self.decision_function(texts)); return np.column_stack([1 - p, p])

def fit_components(fit_frame, reference_frame):
    """Fit teacher, NB transfer, augmented style, and old soft text on one fit partition."""
    seconds, arms = {}, {}
    t = time.perf_counter(); arms["style_orig"] = StyleArm([]).fit(fit_frame.text, fit_frame.label, reference_frame.text); seconds["style_orig"] = time.perf_counter() - t
    p = arms["style_orig"].probabilities(reference_frame.text)
    keep = np.maximum(p, 1 - p) >= .995; assigned = (p[keep] >= .5).astype(int)
    t = time.perf_counter(); arms["nb"] = NBTransferModel().fit(list(fit_frame.text), fit_frame.label, list(reference_frame.text.iloc[np.flatnonzero(keep)]), assigned); seconds["nb_transfer"] = time.perf_counter() - t
    arms["nb_stats"] = margin_reference(arms["nb"], reference_frame.text)
    t = time.perf_counter(); arms["style_aug"] = StyleArm(STYLE_COPIES).fit(fit_frame.text, fit_frame.label, reference_frame.text); seconds["style_aug"] = time.perf_counter() - t
    t = time.perf_counter(); arms["old_soft"] = SoftTextArm().fit(fit_frame.text, fit_frame.label, reference_frame.text); seconds["old_soft"] = time.perf_counter() - t
    acceptance = {"verified_rows": len(fit_frame), "eligible_rows": len(reference_frame), "accepted": int(keep.sum()),
                  "accepted_human": int((assigned == 0).sum()), "accepted_AI": int((assigned == 1).sum()),
                  "pseudo_weighted_mass": float(.25 * keep.sum()), "vocabulary_sha256": arms["nb"].vocabulary_sha256_}
    return arms, acceptance, seconds

def view_components(arms, frame):
    out = {}
    for view in STRESS:
        texts = stress_texts(frame.text, view)
        out[view] = {"style_aug": arms["style_aug"].decision_function(texts), "style_orig": arms["style_orig"].decision_function(texts),
                     "old_soft": arms["old_soft"].decision_function(texts), "nb_transfer": standardized_margin(arms["nb"], texts, arms["nb_stats"])}
    return out

def blend_score(comps, style_key, weights):
    return np.column_stack([comps[style_key], comps["old_soft"], comps["nb_transfer"]]) @ np.asarray(weights)

def grid_table(labels, comps, style_key):
    return [{"candidate": name, "weights": w, **{v: float(roc_auc_score(labels, blend_score(comps[v], style_key, w))) for v in STRESS}}
            for name, w in BLEND_GRID.items()]

def select_blend(rows):
    hist = rows[0]  # pure hist is first in grid order
    for row in rows:
        for view in GUARD_VIEWS:
            row[f"{view}_pass"] = row[view] >= hist[view] - GUARD_SLACK
        row["eligible"] = all(row[f"{v}_pass"] for v in GUARD_VIEWS)
    best = max(r["original"] for r in rows if r["eligible"])
    return next(r for r in rows if r["eligible"] and r["original"] == best)

def calibration(labels, p, bins=10):
    labels, p = np.asarray(labels), np.asarray(p)
    index = np.minimum((p * bins).astype(int), bins - 1)
    table = [{"bin": f"[{k / bins:.1f},{(k + 1) / bins:.1f})", "n": int((index == k).sum()),
              "mean_p_ai": float(p[index == k].mean()) if (index == k).any() else None,
              "fraction_ai": float(labels[index == k].mean()) if (index == k).any() else None} for k in range(bins)]
    ece = sum(r["n"] * abs(r["mean_p_ai"] - r["fraction_ai"]) for r in table if r["n"]) / len(p)
    return {"brier": float(brier_score_loss(labels, p)), "log_loss": float(log_loss(labels, np.clip(p, 1e-15, 1 - 1e-15))),
            "ece_10_bins": float(ece), "bins": table}

@contextmanager
def guarded(stage):
    try:
        yield
    except Exception as exc:
        record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation), {"validation_auroc": None, "failed_stage": stage},
                   time.perf_counter() - START, artifacts=artifacts, status="failed", error_type=type(exc).__name__)
        raise RuntimeError(f"Stage {stage} failed. See the safe failed record.") from None

In [ ]:
# Synthetic checks: stress views, selection guardrails, tie order, calibration bins.
require(stress_texts(["a\n \nb"], "remove_blank") == ["a\nb"], "Remove-blank fixture failed")
require(stress_texts(["a  b\n\nc"], "collapse") == ["a b c"], "Collapse fixture failed")
wrapped = stress_texts(["one two\n\nthree " + "word " * 30], "wrap80")[0]
require(wrapped.startswith("one two\n\nthree") and wrapped.count("\n\n") == 1 and max(map(len, wrapped.split("\n"))) <= 80, "wrap80 fixture failed")
require(normalize_text("\uff21\u201cHi\u201d\u2014x\u200b\n y\u2026") == 'a"hi"-x y...', "Normalization fixture failed")
toy = [{"candidate": n, "original": o, "remove_blank": r, "collapse": c} for n, o, r, c in
       [("hist", .99, .98, .90), ("a", .995, .975, .95), ("b", .99, .99, .99), ("c", .993, .979, .898)]]
require(select_blend(toy)["candidate"] == "c", "Guardrail fixture failed")
toy = [{"candidate": n, "original": .99, "remove_blank": .98, "collapse": .9} for n in ["hist", "a"]]
require(select_blend(toy)["candidate"] == "hist", "Exact tie must go to pure hist")
cal = calibration([0, 1, 1, 0], [.05, .95, 1., .5])
require(cal["bins"][9]["n"] == 2 and cal["bins"][5]["n"] == 1, "Calibration bins failed")
# Real labeled text: every stress view keeps the word sequence.
labeled = pd.concat([train, validation])
for view in STRESS:
    texts = pd.Series(stress_texts(labeled.text, view), index=labeled.index)
    require((texts.str.split() == labeled.text.str.split()).all(), f"{view} changed words")
print("Synthetic and word-sequence checks passed.")

## Run

Save the plan, fit all components on internal 6400, score internal 1600 under four views, select, and save the choice. Only then refit on 8000 and score frozen validation. Any failure writes a safe failed record.

In [ ]:
plan_path = ROOT / "results" / NAME / f"{RUN_ID}-plan.json"
save_new_json(plan_path, {"config": CONFIG, "config_sha256": value_hash(CONFIG), "saved_before_fit": True, "split_sha256": parts["split"]["split_sha256"]})
artifacts, timings = [plan_path], {}
with guarded("internal"), threadpool_limits(limits=1), warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    t = time.perf_counter()
    internal_arms, internal_acceptance, internal_fit_seconds = fit_components(internal_train, reference)
    internal_comps = view_components(internal_arms, internal_score)
    aug_grid = grid_table(internal_score.label, internal_comps, "style_aug")
    orig_grid = grid_table(internal_score.label, internal_comps, "style_orig")
    selected = select_blend(aug_grid)
    orig_selected = select_blend(orig_grid)
    timings["internal_seconds"] = time.perf_counter() - t
    selection_path = ROOT / "results" / NAME / f"{RUN_ID}-selection.json"
    save_new_json(selection_path, {"selected": selected, "grid": aug_grid, "original_style_grid": orig_grid,
        "original_style_selected": orig_selected["candidate"], "acceptance": internal_acceptance,
        "old_rule_active": internal_arms["old_soft"].rule_active_, "old_specialist_rows": internal_arms["old_soft"].specialist_rows_,
        "validation_inspected": False, "utc": datetime.now(timezone.utc).isoformat()})
    artifacts.append(selection_path)
print(f"Selected on internal rows before validation: {selected['candidate']} {selected['weights']}")

In [ ]:
with guarded("full"), threadpool_limits(limits=1), warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    t = time.perf_counter()
    full_arms, full_acceptance, full_fit_seconds = fit_components(train, reference)
    timings["full_fit_seconds"] = time.perf_counter() - t
    final_blend = TransferBlend(full_arms["style_aug"], full_arms["old_soft"], full_arms["nb"], full_arms["nb_stats"], selected["weights"])
    t = time.perf_counter()
    val_comps = view_components(full_arms, validation)
    val_aug_grid = grid_table(validation.label, val_comps, "style_aug")    # descriptive only
    val_orig_grid = grid_table(validation.label, val_comps, "style_orig")  # reproduction of #15 only
    confirmation, val_prob = [], {}
    for view in STRESS:
        val_prob[view] = expit(blend_score(val_comps[view], "style_aug", selected["weights"]))
        confirmation.append({"view": view, **evaluate_predictions(validation, pd.DataFrame({"id": validation.id, "p_ai": val_prob[view]}))})
        path = private / f"blend_validation_{view}.csv"
        pd.DataFrame({"id": validation.id, "p_ai": val_prob[view]}).to_csv(path, index=False); artifacts.append(path)
    # End-to-end model must match the component path.
    end_to_end = class1_probabilities(final_blend, validation.text)
    path_difference = float(np.max(np.abs(end_to_end - val_prob["original"])))
    require(path_difference <= 1e-12, "Model and component paths differ")
    train_prob = class1_probabilities(final_blend, train.text)
    timings["confirmation_seconds"] = time.perf_counter() - t
    path = private / "blend_train.csv"; pd.DataFrame({"id": train.id, "p_ai": train_prob}).to_csv(path, index=False); artifacts.append(path)
    # Standardized component scores for the #16 neural blend step. Private only.
    for part, frame in [("unlabeled", reference), ("test", test)]:
        comps = final_blend.components(frame.text)
        path = private / f"components_{part}.csv"
        pd.DataFrame({"id": frame.id, "style_aug": comps[:, 0], "old_soft": comps[:, 1], "nb_transfer": comps[:, 2],
                      "blend_score": comps @ final_blend.weights}).to_csv(path, index=False); artifacts.append(path)
    for view in STRESS:
        path = private / f"components_validation_{view}.csv"
        pd.DataFrame({"id": validation.id, **{k: val_comps[view][k] for k in ["style_aug", "old_soft", "nb_transfer"]},
                      "blend_score": blend_score(val_comps[view], "style_aug", selected["weights"])}).to_csv(path, index=False); artifacts.append(path)
    model_path = private / "selected_blend.pkl"
    with model_path.open("wb") as handle: cloudpickle.dump(final_blend, handle)
    with model_path.open("rb") as handle: restored = cloudpickle.load(handle)
    reload_difference = float(np.max(np.abs(class1_probabilities(restored, validation.text) - val_prob["original"])))
    require(reload_difference <= 1e-12, "Reload changed probabilities")
    artifacts.append(model_path)
print(f"Confirmed {selected['candidate']} on frozen validation. Reload difference {reload_difference:.3g}")

In [ ]:
# Notebook 08 diagnostics on the held-out selected model (8000 fit, frozen 2000 scored).
with guarded("diagnostics"):
    t = time.perf_counter()
    saved = {"train": pd.DataFrame({"id": train.id, "p_ai": train_prob}), "validation": pd.DataFrame({"id": validation.id, "p_ai": val_prob["original"]})}
    measures = {"train": measure(train), "validation": measure(validation)}
    edges = {column: quartile_edges(measures["train"][column]) for column in DIAG_CONFIG["measures"]}
    overall = {name: evaluate_predictions(frame[["id", "label"]], saved[name]) for name, frame in [("train", train), ("validation", validation)]}
    overall["validation"]["ci_low"], overall["validation"]["ci_high"] = stratified_auc_interval(validation.label, val_prob["original"], DIAG_CONFIG["bootstrap"]["draws"], SEED)
    subgroups = pd.concat([subgroup_table(train, saved["train"], measures["train"], edges, "train", draws=0),
                           subgroup_table(validation, saved["validation"], measures["validation"], edges, "validation",
                                          draws=DIAG_CONFIG["bootstrap"]["draws"], seed=SEED)], ignore_index=True)
    cases, counts = confident_errors(validation, saved["validation"], measures["validation"], DIAG_CONFIG["threshold"], DIAG_CONFIG["cases_per_type"])
    # Error case IDs and scores stay private; only counts and mean measures go in Git.
    path = private / "error_cases.csv"; cases.to_csv(path, index=False); artifacts.append(path)
    case_summary = cases.groupby("error")[["words", "line_breaks", "upper_per_letter", "punct_per_char"]].mean().reset_index()
    calibration_rows = {view: calibration(validation.label, val_prob[view]) for view in ["original", "collapse"]}
    timings["diagnostics_seconds"] = time.perf_counter() - t
timings["run_seconds"] = time.perf_counter() - START

## Results

Internal rows are 1600 rows inside the original 8000. Validation is the frozen 2000, reused before, so optimistic. Probabilities are sigmoid scores; the calibration table checks them instead of assuming. Not Kaggle scores.

In [ ]:
show = lambda rows: pd.DataFrame(rows).drop(columns=["weights"]).set_index("candidate").round(6)
print("Internal 1600, augmented style (selection grid):")
display(show(aug_grid))
print(f"Selected: {selected['candidate']} {selected['weights']}. Guardrail floors: remove_blank >= {aug_grid[0]['remove_blank'] - GUARD_SLACK:.6f}, collapse >= {aug_grid[0]['collapse'] - GUARD_SLACK:.6f}")
print("Internal 1600, original #13 style (reproduces #15, not used for selection):")
display(show(orig_grid))
ref15 = {r["candidate"]: r for r in dep15["metrics"]["internal_blend_grid"]}
repro_internal = max(abs(r[v] - ref15[r["candidate"]][v]) for r in orig_grid for v in ["original", "remove_blank", "collapse"])
ref14 = dep14["metrics"]["internal6400_eval1600_auroc"]["plus_collapse"]
repro_style = {v: aug_grid[0][v] - ref14[v] for v in ["original", "collapse", "wrap80"]}
print(f"Max difference vs #15 internal grid: {repro_internal:.3g}. Pure augmented hist minus #14 plus_collapse: {repro_style}")
acc15 = dep15["metrics"]["internal_transfer_acceptance"]
repro_acceptance = all(internal_acceptance[k] == acc15[k] for k in internal_acceptance)
print(f"Internal pseudo acceptance and NB vocabulary match #15: {repro_acceptance}")
print(f"#15 selected {dep15['metrics']['selected_internal_blend']['candidate']}; our original-style grid selects {orig_selected['candidate']}")
display(pd.DataFrame([{"fit": "internal6400", **internal_acceptance}, {"fit": "development8000", **full_acceptance}]).drop(columns="vocabulary_sha256"))

print("Frozen 2000 validation, selected blend (fit 8000):")
display(pd.DataFrame(confirmation).set_index("view").round(6))
print("Frozen 2000, all candidates, descriptive only (augmented style):")
display(show(val_aug_grid))
print("Frozen 2000, all candidates, original style (reproduces #15 confirmation):")
display(show(val_orig_grid))
ref15_val = {r["variant"]: r["auroc"] for r in dep15["metrics"]["frozen_confirmation"]}
row15 = next(r for r in val_orig_grid if r["candidate"] == dep15["metrics"]["selected_internal_blend"]["candidate"])
repro_val = max(abs(row15[v] - ref15_val[v]) for v in ref15_val)
print(f"Max difference vs #15 frozen confirmation: {repro_val:.3g}")

print("Notebook 08 diagnostics, selected blend on frozen 2000:")
display(pd.DataFrame([{"partition": k, **v} for k, v in overall.items()]).round(6))
vt = subgroups[subgroups.partition == "validation"].drop(columns="partition")
display(vt.merge(subgroups[subgroups.partition == "train"][["measure", "bin", "auroc"]].rename(columns={"auroc": "train_auroc"}), on=["measure", "bin"]).round(6))
print("Confusion at .5:", counts)
display(case_summary.round(4))
for view, cal in calibration_rows.items():
    print(f"Calibration on {view}: Brier {cal['brier']:.6f}, log loss {cal['log_loss']:.6f}, ECE10 {cal['ece_10_bins']:.6f}")
display(pd.DataFrame(calibration_rows["original"]["bins"]).round(4))
display(pd.DataFrame([{"stage": k, "seconds": v} for k, v in {**{f"internal_fit_{k}": s for k, s in internal_fit_seconds.items()},
        **{f"full_fit_{k}": s for k, s in full_fit_seconds.items()}, **timings}.items()]).round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, rows) in zip(axes[:2], [("Internal 1600, augmented style", aug_grid), ("Internal 1600, original style", orig_grid)]):
    for view in STRESS:
        ax.plot([r["candidate"] for r in rows], [r[view] for r in rows], marker="o", label=view)
    ax.set(title=title, ylabel="AUROC"); ax.tick_params(axis="x", rotation=60); ax.legend()
bins = pd.DataFrame(calibration_rows["original"]["bins"]).dropna()
axes[2].plot([0, 1], [0, 1], color="gray", linestyle=":")
axes[2].plot(bins.mean_p_ai, bins.fraction_ai, marker="o")
axes[2].set(title="Frozen 2000 reliability, selected blend", xlabel="mean P(AI) in bin", ylabel="fraction AI")
fig.tight_layout()
plot_path = ROOT / "results" / NAME / f"{RUN_ID}-blend.png"
fig.savefig(plot_path, dpi=130); artifacts.append(plot_path)
plt.show()

## Takeaway

Written after the run from measured numbers.

## Save

Models, row predictions, component scores, and error cases stay in ignored `artifacts/17_final_blend/`. The safe record keeps every grid, the selection, confirmation, diagnostics, calibration, and timings. `refit_spec.json` gives #18 the schema, settings, weights, rule state, and reference statistics. Nothing is uploaded.

In [ ]:
refit_spec = {"selected": selected["candidate"], "weights": selected["weights"], "component_order": CONFIG["components"],
    "style_schema": SCHEMA, "style_schema_sha256": SCHEMA_SHA256, "style": CONFIG["style"], "old_soft": CONFIG["old_soft"],
    "old_soft_models": {"word": {"view": "word_char", "learner": "lr", "C": 16.}, "specialist": {"view": "casechar", "learner": "svm", "C": 1.},
                        "views": EXPANDED_VIEWS, "learners": EXPANDED_LEARNERS},
    "nb_transfer": CONFIG["nb_transfer"], "stress_definitions": CONFIG["stress_definitions"],
    "fit_order": "manifest training order from configs/split.json; reference = eligible unlabeled in split order",
    "threads": CONFIG["threads"], "seed": SEED,
    "development8000_state": {"old_rule_active": full_arms["old_soft"].rule_active_, "old_specialist_rows": full_arms["old_soft"].specialist_rows_,
                              "acceptance": full_acceptance},
    "development8000_reference_stats": {"style_aug": full_arms["style_aug"].statistics_, "style_teacher": full_arms["style_orig"].statistics_,
        "word": full_arms["old_soft"].word_stats_, "specialist": full_arms["old_soft"].specialist_stats_,
        "old_outer": full_arms["old_soft"].outer_, "nb": full_arms["nb_stats"]},
    "note": "Reference stats are for the 8000 fit. A full 10000 refit must recompute every statistic on eligible unlabeled rows."}
spec_path = private / "refit_spec.json"; save_new_json(spec_path, refit_spec); artifacts.append(spec_path)
CONFIG["extra_versions"] = {k: importlib.metadata.version(k) for k in ["cloudpickle", "matplotlib", "threadpoolctl"]}
metrics = {"validation_auroc": confirmation[0]["auroc"], "train_auroc": overall["train"]["auroc"], "selected": selected,
    "internal_grid": aug_grid, "internal_original_style_grid": orig_grid, "original_style_selected": orig_selected["candidate"],
    "frozen_confirmation": confirmation, "validation_grid_descriptive": val_aug_grid, "validation_original_style_grid": val_orig_grid,
    "reproduction": {"max_internal_difference_vs_15": repro_internal, "max_validation_difference_vs_15": repro_val,
                     "pure_aug_hist_minus_14_plus_collapse": repro_style, "internal_acceptance_matches_15": repro_acceptance},
    "internal_acceptance": internal_acceptance, "full_acceptance": full_acceptance,
    "internal_old_rule_active": internal_arms["old_soft"].rule_active_, "full_old_rule_active": full_arms["old_soft"].rule_active_,
    "diagnostics": {"overall": overall, "edges": {c: [None if not np.isfinite(x) else float(x) for x in b] for c, b in edges.items()},
                    "subgroups": json.loads(subgroups.to_json(orient="records")), "confusion_at_threshold": counts,
                    "error_case_mean_measures": json.loads(case_summary.to_json(orient="records"))},
    "calibration": calibration_rows, "fit_seconds": {"internal": internal_fit_seconds, "full": full_fit_seconds}, "timings": timings,
    "model_component_path_difference": path_difference, "max_reload_difference": reload_difference}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation), metrics, time.perf_counter() - START, artifacts=artifacts)
pointer = {"result": str(record_path.relative_to(ROOT)), "result_sha256": file_hash(record_path),
    "split_sha256": parts["split"]["split_sha256"], "selected": selected["candidate"], "weights": selected["weights"],
    "model": str(model_path.relative_to(ROOT)), "refit_spec": str(spec_path.relative_to(ROOT)),
    "validation_predictions": str((private / "blend_validation_original.csv").relative_to(ROOT)),
    "train_predictions": str((private / "blend_train.csv").relative_to(ROOT)),
    "component_scores": {p: str((private / f"components_{p}.csv").relative_to(ROOT)) for p in ["unlabeled", "test"] + [f"validation_{v}" for v in STRESS]},
    "class1_interface": "class1_probabilities(cloudpickle.load(handle), texts)",
    "artifacts": {str(Path(p).relative_to(ROOT)): file_hash(p) for p in artifacts}}
target = ROOT / "artifacts" / NAME / "latest.json"; temporary = target.with_suffix(".tmp")
temporary.write_text(json.dumps(pointer, sort_keys=True, indent=2) + "\n"); temporary.replace(target)
print(f"Saved {record_path.relative_to(ROOT)}")